###Overview
Liquid Clustering: Databricks' next-generation data clustering feature that automatically manages physical data organization for Delta tables.

####What is Liquid Clustering?
- Liquid Clustering is a data layout optimization technique for Delta tables.
- It automatically manages clustering without requiring manual Z-order or partitioning.
- Data is physically organized on disk to minimize scan cost for frequently queried columns.
- It provides better performance for selective queries and incremental updates.
####Key Points
- Introduced in Databricks Runtime 13.3+
- Works with Delta tables only
- Replaces Manual partitioning and Z-Ordering
- Physically reorders data? Yes, in background compaction and OPTIMIZE runs
- Maintenance Databricks handles clustering maintenance automatically
####Typical Use Cases
- Large tables with frequent inserts, updates, and deletes.
- Query filtering on specific columns like customer_id, region, order_date.
- Scenarios where manual ZORDER tuning is difficult or costly.

In [0]:
%sql
-- Step 1: Create a Delta table using Liquid Clustering
-- CLUSTER BY enables liquid clustering automatically.
-- This organizes data dynamically based on chosen columns,
-- improving query performance without rigid partitioning.

CREATE TABLE IF NOT EXISTS new_catalog.default_schema.sales_orders_liquid
(
  order_id INT,        -- Unique order identifier
  customer_id INT,     -- Customer placing the order
  region STRING,       -- Geographic region
  product STRING,      -- Product name
  quantity INT,        -- Quantity ordered
  price DOUBLE,        -- Price per unit
  order_date DATE      -- Date of the order
)
USING DELTA
CLUSTER BY (customer_id, region); -- Liquid clustering keys

In [0]:
%sql
-- Step 2: Insert sample data (multiple small batches)
-- Each insert simulates separate ingestion events.
-- Delta Lake will create new Parquet files for each batch.

-- Batch 1: Initial orders
INSERT INTO new_catalog.default_schema.sales_orders_liquid VALUES
 (1, 101, 'North', 'Laptop', 2, 65000, '2025-10-01'),
 (2, 102, 'South', 'Headphones', 5, 2500, '2025-10-01'),
 (3, 103, 'West', 'Desk Chair', 3, 4500, '2025-10-02');

-- Batch 2: Additional orders
INSERT INTO new_catalog.default_schema.sales_orders_liquid VALUES
 (4, 101, 'North', 'Keyboard', 1, 1200, '2025-10-03'),
 (5, 104, 'East', 'Monitor', 2, 9500, '2025-10-03'),
 (6, 105, 'South', 'Mouse', 4, 700, '2025-10-03');

In [0]:
%sql
-- Step 3: Query data
-- This selects all records from the sales_orders_liquid table.
-- Useful for verifying inserts and inspecting current table contents.

SELECT * 
FROM new_catalog.default_schema.sales_orders_liquid;

In [0]:
%sql
-- Step 4: View table details
-- DESCRIBE DETAIL returns metadata about the Delta table.
-- Includes: catalog, schema, table type, provider, location,
-- row count, file count, size in bytes, and last modification time.
-- Useful for auditing, monitoring ingestion impact, and performance tuning.

DESCRIBE DETAIL new_catalog.default_schema.sales_orders_liquid;

In [0]:
%sql
-- Step 5: View Delta table history
-- DESCRIBE HISTORY shows the full transaction log for the table.
-- Includes: version number, timestamp, user, operation type (CREATE, INSERT, UPDATE, OPTIMIZE, VACUUM).
-- Useful for auditing, debugging, and demonstrating time travel capabilities.

DESCRIBE HISTORY new_catalog.default_schema.sales_orders_liquid;

In [0]:
%sql
-- Step 6: Simulate updates (which trigger reclustering under the hood)
UPDATE new_catalog.default_schema.sales_orders_liquid
SET price = price * 1.05
WHERE region = 'North';
     

In [0]:
%sql
-- Step 6: Verify table history
-- DESCRIBE HISTORY shows the full transaction log for the Delta table.
-- You will see multiple operations here:
--   - CREATE TABLE (initial definition)
--   - INSERT (batch data loads)
--   - UPDATE (price adjustments, reclustering)
--   - Any OPTIMIZE or VACUUM if executed
-- Each entry includes version number, timestamp, user, operation type, and parameters.

DESCRIBE HISTORY new_catalog.default_schema.sales_orders_liquid;

In [0]:
%sql
-- Step 7: Apply targeted updates
-- This modifies rows in the sales_orders_liquid table.
-- Delta Lake creates new Parquet file versions to reflect the changes.
-- Liquid Clustering automatically reclusters data under the hood
-- when updates affect clustering keys or distribution.

UPDATE new_catalog.default_schema.sales_orders_liquid
SET price = price * 1.05       -- Apply a 5% price increase
WHERE region = 'South';        -- Only update orders from the South region

In [0]:
%sql
-- Step 8: Delete records
-- This removes rows from the sales_orders_liquid table.
-- Delta Lake creates new Parquet file versions to reflect the deletion.
-- Liquid Clustering automatically reclusters data under the hood
-- when deletions affect clustering keys or distribution.

DELETE FROM new_catalog.default_schema.sales_orders_liquid
WHERE region = 'East';   -- Remove all orders from the East region

In [0]:
%sql
-- Step 9: Verify Delta table history
-- DESCRIBE HISTORY shows the full transaction log for the table.
-- You will now see multiple operations recorded:
--   - CREATE TABLE (initial definition)
--   - INSERT (batch data loads)
--   - UPDATE (price adjustments for North and South regions)
--   - DELETE (removal of East region orders)
-- Each entry includes version number, timestamp, user, operation type, and parameters.

DESCRIBE HISTORY new_catalog.default_schema.sales_orders_liquid;